# Binoculars over the Palimpsest corpus

Runs cross-perplexity detection on a free Colab T4, because the development laptop has 8 GB of RAM and the model pair does not fit in it. This is the experiment `docs/11-build-and-limits.md` §6.5 lists as *"written and has never been executed"*.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. On CPU this takes hours instead of minutes.

### What this does and does not decide

Binoculars divides a sentence's perplexity under an observer model by the *cross*-perplexity between that model and a closely-related performer:

$$B(s) = \frac{\log \mathrm{PPL}_{\text{obs}}(s)}{\log \mathrm{X\text{-}PPL}(\text{obs}, \text{perf})}$$

Lower means more machine-like. The denominator is the point: plain human writing has low absolute perplexity too — which is *why* raw-perplexity thresholds accuse non-native writers — but two models still disagree about the next word in the ordinary way, so the ratio stays high. Machine text sits where both models agree **and** the realised tokens were unsurprising.

Nothing here is fitted. There is no training set and no threshold learned from our generators, which is the entire reason to reach for it: `docs/08-cross-vendor.md` records a *fitting* failure, and a statistic that fits nothing cannot fail that way.

**Expect a negative result to be possible.** Published Binoculars numbers also degrade on frontier prose. This is worth running precisely because it can come back saying "no".

## 1. Check the GPU you were actually given

Colab hands out whatever is free. The model pair you can afford depends on it, so this is checked rather than assumed.

In [ ]:
import torch, subprocess

if not torch.cuda.is_available():
    print('NO GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.')
    print('You can continue on CPU, but budget hours rather than minutes.')
else:
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'{name}  |  {vram:.1f} GB VRAM')
    print()
    if vram >= 38:
        print("A100-class: run FALCON_7B, the paper's own pair. No excuses about pair size.")
    elif vram >= 14:
        print('T4/L4-class: QWEN_1_5B comfortably, or FALCON_7B with load_in_8bit=True.')
    else:
        print('Small GPU: stay on QWEN_0_6B (the pair already proven to run).')

In [ ]:
!pip -q install "transformers>=4.44" accelerate bitsandbytes
print('ok')

## 2. Upload the bundle

Produced locally by `python scripts/export_for_colab.py`. Upload **both** files from `colab_bundle/`:

* `documents.jsonl` — essays plus the exact sentence spans to score
* `palimpsest_binoculars.py` — the repository's own scorer

The scorer is uploaded rather than pasted into a cell on purpose. This project's recurring bug is a fix applied to one copy of two; a notebook with the maths retyped into it *is* a second copy.

**The essays are real student writing, and uploading them sends the text to Google.** That is a governance decision, not a technical one — see the note in `export_for_colab.py`.

In [ ]:
import os, json, shutil

NEED = ['documents.jsonl', 'palimpsest_binoculars.py']
DRIVE_DIR = '/content/drive/MyDrive/palimpsest'   # where you put the two files

# Google Drive first. A free Colab session can be reclaimed at any time, and the
# files.upload() widget makes you re-send 12 MB every time that happens. Drive also gives
# the run somewhere durable to CHECKPOINT to, so a disconnect at minute 18 of 20 costs
# nothing instead of the whole run.
WORK = '/content'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    if all(os.path.exists(f'{DRIVE_DIR}/{f}') for f in NEED):
        WORK = DRIVE_DIR
        print(f'using Drive: {WORK}')
    else:
        print(f'Drive mounted but {DRIVE_DIR} does not hold both files.')
        print('Either put them there, or use the upload widget below.')
except Exception as exc:
    print(f'Drive unavailable ({type(exc).__name__}); falling back to the upload widget.')

# Fallback: the upload widget, for anything still missing.
missing = [f for f in NEED if not os.path.exists(f'{WORK}/{f}')]
if missing:
    from google.colab import files
    print('\nupload:', ', '.join(missing))
    files.upload()
    for f in missing:
        if os.path.exists(f) and WORK != '/content':
            shutil.copy(f, f'{WORK}/{f}')

missing = [f for f in NEED if not os.path.exists(f'{WORK}/{f}')]
assert not missing, f'still missing: {missing}'

# The scorer must be importable, so its directory has to be on the path.
import sys
if WORK not in sys.path:
    sys.path.insert(0, WORK)

docs = [json.loads(l) for l in open(f'{WORK}/documents.jsonl', encoding='utf-8') if l.strip()]
spans = sum(len(d['spans']) for d in docs)
empty = sum(1 for d in docs if not d.get('text'))
print(f'\n{len(docs):,} documents, {spans:,} sentence spans')
print(f'working directory: {WORK}')
assert not empty, (
    f'{empty} documents have no text -- this bundle was built with --no-text, '
    'which cannot score anything. Re-export without that flag.')

## 3. Choose the model pair

**Both models must share a tokenizer.** The denominator is a cross-entropy between two distributions over one vocabulary; with different vocabularies it is not a cross-entropy at all, just two unrelated numbers divided. The scorer asserts this at construction, so a wrong pair fails loudly rather than returning plausible nonsense.

`binoculars.py` notes that published ablations show accuracy falling with pair size, and that the 0.6 B pair is therefore *"a weaker instrument than the paper's"*. On a T4 you can close that gap.

In [ ]:
QWEN_0_6B  = ('Qwen/Qwen3-0.6B-Base',   'Qwen/Qwen3-0.6B')   # proven to run on 8 GB CPU
QWEN_1_5B  = ('Qwen/Qwen2.5-1.5B',      'Qwen/Qwen2.5-1.5B-Instruct')  # free-tier T4
FALCON_7B  = ('tiiuae/falcon-7b',       'tiiuae/falcon-7b-instruct')   # the paper's pair

OBSERVER, PERFORMER = QWEN_1_5B   # <- change this line

# NO QUANTIZATION OPTION, deliberately. An earlier draft of this notebook offered a
# LOAD_IN_8BIT flag and it was dead code: BinocularsScorer takes no quantization
# parameter, so the flag silently did nothing. It is removed rather than wired up.
# The reason is in binoculars.py, which already takes every softmax in float32 because
# "bf16 has ~3 decimal digits, and the ratio is a quotient of two averaged logs where
# that error would not cancel". Quantising the weights underneath that is the same
# objection one level down. If FALCON_7B does not fit, run a smaller pair and report
# WHICH pair produced the number -- do not quantise and still call it Falcon.
vram = (torch.cuda.get_device_properties(0).total_memory / 1e9
        if torch.cuda.is_available() else 0.0)
NEEDS = {'tiiuae/falcon-7b': 15.5, 'Qwen/Qwen2.5-1.5B': 4.0, 'Qwen/Qwen3-0.6B-Base': 2.0}
want = NEEDS.get(OBSERVER, 4.0)

print(f'observer  {OBSERVER}')
print(f'performer {PERFORMER}')
print(f'the pair needs ~{want:.1f} GB; this GPU has {vram:.1f} GB')
if vram and want > vram:
    print('\n!! This pair will not fit. Choose a smaller one above, or use Colab Pro for')
    print('   an A100/L4. Do not quantise to force it -- see the comment in this cell.')
elif vram:
    print('fits')

In [ ]:
import importlib, palimpsest_binoculars as pb
importlib.reload(pb)

scorer = pb.BinocularsScorer(observer=OBSERVER, performer=PERFORMER, device='auto')
print('loaded on', scorer.device, '|', scorer.dtype)

# Sanity check on two spans whose answer we already know from the CPU run: machine-ish
# prose must score BELOW human-ish prose. If this ordering is wrong the run is not worth
# starting -- something about the pair or the dtype is broken.
m = scorer.score('Throughout my academic journey, I have consistently sought opportunities '
                 'to challenge myself and grow as both a student and a leader. This '
                 'experience taught me the importance of perseverance and collaboration.')
h = scorer.score('My grandmother kept her rosary in a chipped blue bowl by the sink. I never '
                 'asked why there and not the drawer. She died in March; the bowl is still there.')
print(f'machine-ish B = {m.score:.4f}')
print(f'human-ish   B = {h.score:.4f}')
print('ordering', 'OK' if m.score < h.score else 'WRONG -- investigate before scoring')

## 4. Score every document

One forward pass per document per model; each sentence is then a **slice** of that token stream, taken with the scorer's own `select()` so token-to-sentence attribution matches the rest of the pipeline exactly.

Two things worth knowing about the arithmetic:

* The Binoculars score of a span is a **ratio of means**, not the mean of per-token ratios. Those differ, and only the first is the published statistic — which is why `BinocularsScores` stores the two sums' ingredients per token rather than a per-token ratio.
* `logppl` and `xppl` are exported alongside the ratio so the division can be audited later instead of taken on trust.

Checkpoints every 200 documents, because a Colab session can be reclaimed at any time and losing an hour to a disconnect is avoidable.

In [ ]:
import json, time, math, numpy as np

# Written into WORK, so on Drive the checkpoint survives a disconnected session and the
# run resumes where it stopped rather than starting over.
OUT = f'{WORK}/binoculars_scores.jsonl'
CHECKPOINT_EVERY = 200

done = set()
try:
    for line in open(OUT, encoding='utf-8'):
        if line.strip():
            done.add(json.loads(line)['doc_id'])
    print(f'resuming: {len(done):,} documents already scored')
except FileNotFoundError:
    pass

todo = [d for d in docs if d['doc_id'] not in done]
print(f'{len(todo):,} to go  ->  {OUT}')

t0, failed = time.time(), []
with open(OUT, 'a', encoding='utf-8') as fh:
    for n, d in enumerate(todo, 1):
        try:
            tokens = scorer.score(d['text'])
            rows = []
            for sp in d['spans']:
                sub = tokens.select(sp['start'], sp['end'])
                b = sub.score
                # NaN is the honest answer for a span too short to estimate. It is written
                # as null and imputed to the training mean downstream, contributing exactly
                # zero to the logit -- never silently replaced with a number.
                lp = float(-np.mean(sub.logprob)) if len(sub.logprob) else float('nan')
                xp = float(np.mean(sub.xent)) if len(sub.xent) else float('nan')
                rows.append({
                    'i': sp['i'],
                    'binoculars_score': None if math.isnan(b) else round(b, 6),
                    'binoculars_logppl': None if math.isnan(lp) else round(lp, 6),
                    'binoculars_xppl': None if math.isnan(xp) else round(xp, 6),
                    'n_tokens': len(sub),
                })
            fh.write(json.dumps({'doc_id': d['doc_id'], 'sentences': rows}) + '\n')
        except Exception as exc:
            # One bad document must not end a twenty-minute run; it is recorded and skipped.
            # join_binoculars.py will REFUSE the set if any document is missing, so a
            # failure here surfaces later as a refusal rather than as a silent gap.
            failed.append((d['doc_id'], f'{type(exc).__name__}: {exc}'[:140]))

        if n % CHECKPOINT_EVERY == 0 or n == len(todo):
            fh.flush()
            rate = n / max(time.time() - t0, 1e-9)
            left = (len(todo) - n) / max(rate, 1e-9)
            print(f'{n:,}/{len(todo):,}  {rate:.1f} docs/s  ~{left/60:.0f} min left')

print(f'\ndone in {(time.time() - t0) / 60:.1f} min')
if failed:
    print(f'\n{len(failed)} documents FAILED and are absent from the output.')
    print('The join will refuse the affected set until these are scored or excluded:')
    for doc_id, err in failed[:10]:
        print(' ', doc_id, err)
else:
    print('no failures')

## 5. Look at it before downloading

A separation here is not a result — these are the scores this corpus produces, and the corpus has known confounds (`docs/12`: a 0.960 AUROC that fell to 0.490 on somebody else's essays). It is a smoke test: if machine and human medians are identical, something is wrong with the run, not with the world.

In [ ]:
import numpy as np, json, collections

src = {d['doc_id']: d['source'] for d in docs}
by_source = collections.defaultdict(list)
n_null = n_total = 0

for line in open(OUT, encoding='utf-8'):
    if not line.strip():
        continue
    r = json.loads(line)
    vals = [s['binoculars_score'] for s in r['sentences']]
    n_total += len(vals)
    n_null += sum(1 for v in vals if v is None)
    good = [v for v in vals if v is not None]
    if good:
        by_source[src.get(r['doc_id'], '?')].append(float(np.mean(good)))

print(f'{n_total:,} spans, {n_null:,} unmeasurable ({n_null/max(n_total,1):.1%} '
      '-- short spans, which is expected)\n')
print(f"{'source':34} {'n':>5}  {'median B':>9}")
for s, v in sorted(by_source.items(), key=lambda kv: np.median(kv[1])):
    print(f'{s:34} {len(v):5}  {np.median(v):9.4f}')
print('\nLower = more machine-like. Machine sources should sort to the TOP.')

In [ ]:
import os

if WORK.startswith('/content/drive'):
    print(f'Already saved to Drive:\n  {OUT}')
    print('\nDownload it from drive.google.com (folder: MyDrive/palimpsest),')
    print('or run the line below to pull it straight down.')

try:
    from google.colab import files
    files.download(OUT)
except Exception as exc:
    print(f'(browser download unavailable: {type(exc).__name__}) '
          f'-- the file is at {OUT}')

print(f'\nsize: {os.path.getsize(OUT) / 1e6:.1f} MB')
print('\nThen, in the repository:')
print('  python scripts/join_binoculars.py --scores ~/Downloads/binoculars_scores.jsonl --dry-run')
print('  python scripts/join_binoculars.py --scores ~/Downloads/binoculars_scores.jsonl')
print('  python scripts/syntax_probe.py     # re-measure with the new column')